# Dataset sizes


In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.family"] = "Helvetica"
plt.rcParams["text.color"] = "black"
plt.rcParams["axes.labelcolor"] = "black"
plt.rcParams["xtick.color"] = "black"
plt.rcParams["ytick.color"] = "black"
plt.rcParams["axes.edgecolor"] = "black"

figure_dir = "figures"

dataset_df = pd.read_csv(
    "https://raw.githubusercontent.com/sdorkenw/connectomics_review/main/dataset_sizes_export_261121.csv"
)
dataset_df["Publication date"] = pd.to_datetime(dataset_df["Publication date"])
days_in_year = np.where(dataset_df["Publication date"].dt.is_leap_year, 366, 365)
dataset_df["Year-fractional"] = (
    dataset_df["Publication date"].dt.year
    + dataset_df["Publication date"].dt.dayofyear / days_in_year
)
dataset_df = dataset_df.rename(
    columns={"Volume (mm3)": "Volume", "Related-ID": "Relation"}
).set_index("ID")


def plot_break(scatter, stem):
    fig, (ax1, ax2) = plt.subplots(
        1, 2, sharey=True, figsize=(7, 5.5), dpi=100,
        gridspec_kw={"width_ratios": [0.3, 0.7], "wspace": 0.0},
    )
    xlims = ([1970, 2010], [2010, 2028])

    for ax in (ax1, ax2):
        ax.tick_params(length=4, width=2, labelsize=22, which="major")
        ax.tick_params(length=0, width=0, labelsize=22, which="minor")
        ax.spines["bottom"].set_linewidth(2)
        ax.spines["top"].set_linewidth(0)
        ax.spines["right"].set_linewidth(0)
    ax1.spines["left"].set_linewidth(2)
    ax2.spines["left"].set_linewidth(0)
    ax2.tick_params(left=False)

    for ax, ax_xlim in zip((ax1, ax2), xlims):
        sub = dataset_df[
            (dataset_df["Year-fractional"] >= ax_xlim[0])
            & (dataset_df["Year-fractional"] < ax_xlim[1])
        ]
        scatter(ax, sub)
        for _, row in sub[sub["Relation"] != 0].iterrows():
            x0 = dataset_df.loc[row["Relation"], "Year-fractional"]
            x1 = row["Year-fractional"]
            ax.hlines(row["Volume"], x0, x1, zorder=-1, lw=3, color=".75")
            if row["Relation"] not in sub.index:
                other = ax2 if ax is ax1 else ax1
                other.hlines(row["Volume"], x0, x1, zorder=-1, lw=3, color=".75")

    ax1.set_xlim(xlims[0])
    ax2.set_xlim(xlims[1])
    ax1.set_yscale("log")
    ax2.set_yscale("log")
    ax1.set_xticks([1970, 1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010])
    ax1.set_xticklabels([1970, "", "", "", 1990, "", "", "", ""])
    ax1.set_xlabel("")
    ax1.vlines(2010, 1e-8, 10, lw=2, color=".0", ls="--")
    ax2.vlines(2010, 1e-8, 10, lw=2, color=".0", ls="--")
    ax2.set_xlabel("Publication year", fontsize=24)
    ax1.set_ylabel("Dataset volume ($mm^3$)", fontsize=24)

    plt.tight_layout()
    plt.savefig(f"{figure_dir}/dataset_sizes_pub_{stem}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{figure_dir}/dataset_sizes_pub_{stem}.pdf", dpi=300, bbox_inches="tight")
    plt.show()


## All datasets


In [ ]:
plot_break(
    lambda ax, data: sns.scatterplot(
        data=data, x="Year-fractional", y="Volume",
        marker="o", s=66, lw=0.5, ax=ax, color="k",
    ),
    "blank_break",
)


## Reconstruction


In [ ]:
reconst_cmap = {
    "manual": "k",
    "semi-automated": [0.9, 0.1, 0.2],
}

plot_break(
    lambda ax, data: sns.scatterplot(
        data=data, x="Year-fractional", y="Volume",
        marker="o", s=100, lw=0.5, ax=ax,
        hue="Reconstruction", palette=reconst_cmap, legend=False,
    ),
    "reconstruction_break",
)


## Species


In [ ]:
species_cmap = {
    "Drosophila": [0.1, 0.7, 0.8],
    "Mouse": [0.8, 0.4, 0.1],
    "Zebrafish": [0.7, 0.2, 0.9],
}
for species in dataset_df["Species"].unique():
    species_cmap.setdefault(species, "k")

plot_break(
    lambda ax, data: sns.scatterplot(
        data=data, x="Year-fractional", y="Volume",
        marker="o", s=100, lw=0.5, ax=ax,
        hue="Species", palette=species_cmap, legend=False,
    ),
    "mouse_fly_break",
)
